In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, roc_auc_score

In [2]:
df = pd.read_csv("/content/cyclone_labels.csv")

print(df.shape)

df.head()

(10538, 8)


,Basin,CycloneID,Longitude,Latitude,DateTime,WindSpeed,PressureDrop,Pressure
0,ATLN,200301L,-66.0,31.4,2003041815,30.0,0.0,1008.0
1,ATLN,200301L,-66.3,31.9,2003041818,30.0,0.0,1007.0
2,ATLN,200301L,-66.6,32.5,2003041821,30.0,0.0,1007.0
3,ATLN,200301L,-68.6,34.5,2003041912,35.0,0.0,1006.0
4,ATLN,200301L,-68.8,34.4,2003041915,35.0,0.0,1006.0


In [3]:
df = pd.read_csv("/content/cyclone_labels.csv")

print(df.shape)

df.head()

(10538, 8)


,Basin,CycloneID,Longitude,Latitude,DateTime,WindSpeed,PressureDrop,Pressure
0,ATLN,200301L,-66.0,31.4,2003041815,30.0,0.0,1008.0
1,ATLN,200301L,-66.3,31.9,2003041818,30.0,0.0,1007.0
2,ATLN,200301L,-66.6,32.5,2003041821,30.0,0.0,1007.0
3,ATLN,200301L,-68.6,34.5,2003041912,35.0,0.0,1006.0
4,ATLN,200301L,-68.8,34.4,2003041915,35.0,0.0,1006.0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10538 entries, 0 to 10537
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Basin         10538 non-null  object 
 1   CycloneID     10538 non-null  object 
 2   Longitude     10538 non-null  float64
 3   Latitude      10538 non-null  float64
 4   DateTime      10538 non-null  int64  
 5   WindSpeed     10538 non-null  float64
 6   PressureDrop  10538 non-null  float64
 7   Pressure      10538 non-null  float64
dtypes: float64(5), int64(1), object(2)
memory usage: 658.8+ KB


In [5]:
df.isnull().sum()

,0
Basin,0
CycloneID,0
Longitude,0
Latitude,0
DateTime,0
WindSpeed,0
PressureDrop,0
Pressure,0


In [6]:
df = df.drop(
    ["CycloneID"],
    axis=1
)

df.head()

,Basin,Longitude,Latitude,DateTime,WindSpeed,PressureDrop,Pressure
0,ATLN,-66.0,31.4,2003041815,30.0,0.0,1008.0
1,ATLN,-66.3,31.9,2003041818,30.0,0.0,1007.0
2,ATLN,-66.6,32.5,2003041821,30.0,0.0,1007.0
3,ATLN,-68.6,34.5,2003041912,35.0,0.0,1006.0
4,ATLN,-68.8,34.4,2003041915,35.0,0.0,1006.0


In [7]:
df["DateTime"].head()

,DateTime
0,2003041815
1,2003041818
2,2003041821
3,2003041912
4,2003041915


In [8]:
df["DateTime"] = pd.to_datetime(
    df["DateTime"].astype(str),
    format="%Y%m%d%H"
)

In [9]:
df["year"] = df["DateTime"].dt.year
df["month"] = df["DateTime"].dt.month
df["day"] = df["DateTime"].dt.day

df = df.drop("DateTime", axis=1)

In [10]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

df["Basin"] = encoder.fit_transform(
    df["Basin"]
)

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10538 entries, 0 to 10537
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Basin         10538 non-null  int64  
 1   Longitude     10538 non-null  float64
 2   Latitude      10538 non-null  float64
 3   WindSpeed     10538 non-null  float64
 4   PressureDrop  10538 non-null  float64
 5   Pressure      10538 non-null  float64
 6   year          10538 non-null  int32  
 7   month         10538 non-null  int32  
 8   day           10538 non-null  int32  
dtypes: float64(5), int32(3), int64(1)
memory usage: 617.6 KB


In [12]:
import numpy as np

df["target"] = np.where(
    df["WindSpeed"] >= 34,
    1,
    0
)

In [13]:
df["target"].value_counts()

,count
target,
1,6720
0,3818


In [14]:
df.head()

,Basin,Longitude,Latitude,WindSpeed,PressureDrop,Pressure,year,month,day,target
0,0,-66.0,31.4,30.0,0.0,1008.0,2003,4,18,0
1,0,-66.3,31.9,30.0,0.0,1007.0,2003,4,18,0
2,0,-66.6,32.5,30.0,0.0,1007.0,2003,4,18,0
3,0,-68.6,34.5,35.0,0.0,1006.0,2003,4,19,1
4,0,-68.8,34.4,35.0,0.0,1006.0,2003,4,19,1


In [15]:
df["target"].value_counts()

,count
target,
1,6720
0,3818


In [16]:
X = df.drop("target", axis=1)

y = df["target"]

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [18]:
from lightgbm import LGBMClassifier


cyclone_model = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    random_state=42
)


cyclone_model.fit(
    X_train,
    y_train
)

[LightGBM] [Info] Number of positive: 5376, number of negative: 3054
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001213 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1006
[LightGBM] [Info] Number of data points in the train set: 8430, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.637722 -> initscore=0.565492
[LightGBM] [Info] Start training from score 0.565492
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, bes

LGBMClassifier(learning_rate=0.05, n_estimators=200, random_state=42)

In [19]:
from sklearn.metrics import classification_report, roc_auc_score


y_pred = cyclone_model.predict(X_test)


print(
    classification_report(
        y_test,
        y_pred
    )
)


prob = cyclone_model.predict_proba(X_test)[:,1]

print(
    "ROC-AUC:",
    roc_auc_score(y_test, prob)
)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       764
           1       1.00      1.00      1.00      1344

    accuracy                           1.00      2108
   macro avg       1.00      1.00      1.00      2108
weighted avg       1.00      1.00      1.00      2108

ROC-AUC: 1.0


In [20]:
import joblib

joblib.dump(
    cyclone_model,
    "cyclone_model.pkl"
)

print("Cyclone model saved")

Cyclone model saved
